# Phase 2 - Create Bronze Weather Table

Bronze weather normalizes timestamps, converts scientific units into model-friendly values, selects the required fields, and writes partitioned Parquet to MinIO.

Production counterpart: `spark_jobs/build_bronze_weather.py`.

In [1]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

PROJECT_ROOT = Path("/workspace")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spark_jobs.common import create_spark_session, load_settings, normalized_timestamp, s3a_uri

YEAR, MONTH = 2024, 1
WRITE_TABLE = False  # Set True to rebuild the verified Bronze partition during the demo.
settings = load_settings()
spark = create_spark_session("phase2-create-bronze-weather", settings)
source = s3a_uri(settings.raw_bucket, f"arco_era5_us_airport_hourly/year={YEAR}/month={MONTH:02d}")
destination = s3a_uri(settings.lakehouse_bucket, f"bronze/weather/year={YEAR}/month={MONTH:02d}")
print("Source:", source)
print("Destination:", destination)

Source: s3a://raw/arco_era5_us_airport_hourly/year=2024/month=01
Destination: s3a://lakehouse/bronze/weather/year=2024/month=01


## Visible Bronze Transformation

The source stores temperature in Kelvin, precipitation in metres, and wind components in metres per second. Bronze derives Celsius, millimetres, and knots.

In [2]:
raw_df = spark.read.parquet(source)
bronze_weather_df = (
    raw_df
    .withColumn("time_utc", normalized_timestamp(raw_df, "time_utc"))
    .withColumn("temperature_c", F.col("2m_temperature") - F.lit(273.15))
    .withColumn("dewpoint_temperature_c", F.col("2m_dewpoint_temperature") - F.lit(273.15))
    .withColumn("wind_speed_ms", F.sqrt(F.pow(F.col("10m_u_component_of_wind"), 2) + F.pow(F.col("10m_v_component_of_wind"), 2)))
    .withColumn("wind_speed_kts", F.col("wind_speed_ms") * F.lit(1.94384))
    .withColumn("wind_gust_kts", F.col("10m_wind_gust_since_previous_post_processing") * F.lit(1.94384))
    .withColumn("precipitation_mm", F.col("total_precipitation") * F.lit(1000.0))
    .select(
        F.col("time_utc").cast("timestamp"), F.col("airport_key").cast("int"),
        F.col("day").cast("int"), F.col("hour_utc").cast("int"),
        F.col("temperature_c").cast("double"), F.col("dewpoint_temperature_c").cast("double"),
        F.col("wind_speed_ms").cast("double"), F.col("wind_speed_kts").cast("double"),
        F.col("wind_gust_kts").cast("double"), F.col("precipitation_mm").cast("double"),
        F.col("surface_pressure").cast("double").alias("surface_pressure_pa"),
        F.col("mean_sea_level_pressure").cast("double").alias("mean_sea_level_pressure_pa"),
        F.col("total_cloud_cover").cast("double"),
        F.col("convective_available_potential_energy").cast("double").alias("cape_j_kg"),
    )
)
bronze_weather_df.printSchema()
bronze_weather_df.show(10, truncate=False)

root
 |-- time_utc: timestamp (nullable = true)
 |-- airport_key: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour_utc: integer (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- dewpoint_temperature_c: double (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- wind_speed_kts: double (nullable = true)
 |-- wind_gust_kts: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- surface_pressure_pa: double (nullable = true)
 |-- mean_sea_level_pressure_pa: double (nullable = true)
 |-- total_cloud_cover: double (nullable = true)
 |-- cape_j_kg: double (nullable = true)



+-------------------+-----------+---+--------+------------------+----------------------+------------------+------------------+------------------+--------------------+-------------------+--------------------------+------------------+--------------+
|time_utc           |airport_key|day|hour_utc|temperature_c     |dewpoint_temperature_c|wind_speed_ms     |wind_speed_kts    |wind_gust_kts     |precipitation_mm    |surface_pressure_pa|mean_sea_level_pressure_pa|total_cloud_cover |cape_j_kg     |
+-------------------+-----------+---+--------+------------------+----------------------+------------------+------------------+------------------+--------------------+-------------------+--------------------------+------------------+--------------+
|2024-01-01 00:00:00|10000      |1  |0       |13.927423095703148|9.696679687500023     |0.7493602348399461|1.4566363988912807|5.034089864196777 |0.013416633009910583|101931.59375       |101959.765625             |0.9983519315719604|5.65234375    |
|2024-01

In [3]:
if WRITE_TABLE:
    bronze_weather_df.write.mode("overwrite").parquet(destination)
    print("Rebuilt Bronze weather partition.")
else:
    print("Using the previously verified Bronze output. Set WRITE_TABLE=True to rebuild.")

stored_bronze_df = spark.read.parquet(destination)
print("Stored Bronze weather rows:", stored_bronze_df.count())
stored_bronze_df.select("temperature_c", "wind_speed_kts", "precipitation_mm").summary().show()

Using the previously verified Bronze output. Set WRITE_TABLE=True to rebuild.


Stored Bronze weather rows: 18693744


+-------+-------------------+--------------------+--------------------+
|summary|      temperature_c|      wind_speed_kts|    precipitation_mm|
+-------+-------------------+--------------------+--------------------+
|  count|           18693744|            18693744|            18693744|
|   mean| 2.0028336447109996|   6.582588932249966| 0.14751322978675915|
| stddev|  9.808494285379705|  3.9862879659454307|  0.5751019035887811|
|    min| -52.70932617187498|0.002929010985621998|-1.86264514923095...|
|    25%|-2.9905151367187273|  3.7329992744820504|                 0.0|
|    50%| 2.1195617675781477|   5.715040455541363|1.862645149230957E-6|
|    75%|  8.327844238281273|    8.57467412395239|0.024121254682540894|
|    max|  32.38729248046877|  46.141114847353876|  23.029297590255737|
+-------+-------------------+--------------------+--------------------+



## Verified Bronze Result

The stored January partition contains **18,693,744 rows** and occupies approximately **757 MiB** in MinIO.

In [4]:
spark.stop()